In [0]:
import requests

url = "https://www.imf.org/external/datamapper/api/v1/countries"
response = requests.get(url)
data = response.json()

import pandas as pd
df = pd.DataFrame(data['countries'])
# display(df)

df_unpivoted = df.melt()
# display(df_unpivoted)

df_unpivoted = df_unpivoted.rename(columns={'variable': 'country_code', 'value': 'country_name'})
display(df_unpivoted)

df_unpivoted_spark = spark.createDataFrame(df_unpivoted)
df_unpivoted_spark.write.format("delta").mode("overwrite").saveAsTable("default.imf_countries")

In [0]:
%%sql
Update default.imf_countries
set country_name = 'Bharat'
where country_code = 'IND';
select * from default.imf_countries
where country_code = 'IND'

In [0]:
%sql
 
-- CREATE OR REPLACE TABLE default.imf_countries_scd2 (
--   country_code STRING,
--   country_name STRING,
--   effective_date DATE,
--   end_date DATE,
--   is_current BOOLEAN
-- );

-- Find country codes with updated country names
WITH updated_countries AS (
  SELECT
    c.country_code
  FROM default.imf_countries c
  LEFT JOIN default.imf_countries_scd2 s
    ON c.country_code = s.country_code AND s.is_current = TRUE
  WHERE c.country_name != s.country_name OR s.country_name IS NULL
)
INSERT INTO default.imf_countries_scd2
SELECT
  country_code,
  country_name,
  current_date() AS effective_date,
  NULL AS end_date,
  TRUE AS is_current
FROM default.imf_countries;

-- updated_countries identifies country codes in default.imf_countries where the country_name has changed
-- compared to the current record in default.imf_countries_scd2, or where no current record exists.
-- These are the records that require a new SCD2 instance.
SELECT
  c.country_code
FROM default.imf_countries c
LEFT JOIN default.imf_countries_scd2 s
  ON c.country_code = s.country_code AND s.is_current = TRUE
WHERE c.country_name != s.country_name OR s.country_name IS NULL



In [0]:
%sql

select * from default.imf_countries_scd2
where country_code = 'IND'

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load current SCD2 table
scd2_table = spark.table("default.imf_countries_scd2")

# Prepare new data
new_data = df_unpivoted_spark.withColumn("effective_date", F.current_date()) \
    .withColumn("end_date", F.lit(None).cast("date")) \
    .withColumn("is_current", F.lit(True))

# Join to find changed records
join_cond = [
    scd2_table.country_code == new_data.country_code
]
changed = new_data.join(
    scd2_table.filter("is_current = True"),
    join_cond,
    "left"
).filter(
    (scd2_table.country_name != new_data.country_name) | scd2_table.country_name.isNull()
)

# Expire old records
expired = scd2_table.join(
    changed.select("country_code"),
    "country_code"
).filter("is_current = True") \
 .withColumn("end_date", F.current_date()) \
 .withColumn("is_current", F.lit(False))

# Union unchanged, expired, and new records
unchanged = scd2_table.join(
    changed.select("country_code"),
    "country_code",
    "left_anti"
)

final_scd2 = unchanged.unionByName(expired).unionByName(changed.select(scd2_table.columns))

# Write back to Delta table
final_scd2.write.format("delta").mode("overwrite").saveAsTable("default.imf_countries_scd2")